In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import re
from scipy import stats
import utils
import seaborn as sns
import pingouin as pg
import matplotlib.cm as cm

import statsmodels.stats.power as smp
from statsmodels.stats.anova import AnovaRM
from tqdm import tqdm


from natsort import index_natsorted

# plt.rcParams['font.family'] = 'Times New Roman'
# plt.rcParams['font.family'] = 'Calibri'

path_figs = "./Figs/"

fingers = ['1', '2', '3', '4', '5']

iti = 1000 # msecs for inter-trial interval
planTime = 2000 # msecs for precue time
feedbackTime = 2000 # msecs for feedback time


total_sub_num = 20
num_sessions = 2
num_blocks_per_session = 12
num_trials_per_block = 30

sub_nums = [1,2]

utils.set_figure_style("1col")
sns.color_palette('colorblind')


/Users/amin/projects/LearningDynamics/ChordAdaptationDynamics/Version2/utils.py:138: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
  subj_melted['N'] = (subj_melted['IPI_Number'].str.extract('(\d+)').astype('int64') + 1)
/Users/amin/projects/LearningDynamics/ChordAdaptationDynamics/Version2/utils.py:157: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
  subj_melted['N'] = subj_melted['Press_Number'].str.extract('(\d+)').astype('int64')
/Users/amin/projects/LearningDynamics/ChordAdaptationDynamics/Version2/utils.py:172: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
  subj_melted['N'] = subj_melted['Response_Number'].str.extract('(\d+)').astype('int64')


[(0.00392156862745098, 0.45098039215686275, 0.6980392156862745),
 (0.8705882352941177, 0.5607843137254902, 0.0196078431372549),
 (0.00784313725490196, 0.6196078431372549, 0.45098039215686275),
 (0.8352941176470589, 0.3686274509803922, 0.0),
 (0.8, 0.47058823529411764, 0.7372549019607844),
 (0.792156862745098, 0.5686274509803921, 0.3803921568627451),
 (0.984313725490196, 0.6862745098039216, 0.8941176470588236),
 (0.5803921568627451, 0.5803921568627451, 0.5803921568627451),
 (0.9254901960784314, 0.8823529411764706, 0.2),
 (0.33725490196078434, 0.7058823529411765, 0.9137254901960784)]

In [2]:
# reload utils
import importlib
importlib.reload(utils)

<module 'utils' from '/Users/amin/projects/LearningDynamics/ChordAdaptationDynamics/Version2/utils.py'>

In [3]:
subjs_list = utils.read_dat_files_subjs_list(sub_nums)

subjs = pd.concat(subjs_list, ignore_index=True)

subjs['TotalTrialNum'] = (subjs['BN'] - 1) * num_trials_per_block + subjs['TN']
subjs['day'] = subjs['BN'].apply(lambda x: (x - 1) // num_blocks_per_session + 1)
subjs['chord'] = subjs.apply(utils.extract_chord_form_target_force, axis=1)
subjs['block_type'] = subjs.apply(utils.extract_block_type, axis=1)
subjs['trial_num_within_chord'] = subjs.apply(utils.extract_trial_num_within_chord, axis=1)


subjs['num_targets'] = subjs[[f'targetForce{i}' for i in range(1, 6)]].ne(0).sum(axis=1)
subjs.reset_index(drop=True, inplace=True)



In [4]:
subjs.columns

Index(['BN', 'TN', 'subNum', 'targetForce1', 'targetForce2', 'targetForce3',
       'targetForce4', 'targetForce5', 'endForce1', 'endForce2', 'endForce3',
       'endForce4', 'endForce5', 'endForcePurturbed1', 'endForcePurturbed2',
       'endForcePurturbed3', 'endForcePurturbed4', 'endForcePurturbed5',
       'isTargetVisible', 'purturbation1', 'purturbation2', 'planTime',
       'feedbackTime', 'iti', 'forceGain', 'trialCorr', 'trialErrorType',
       'TotalTrialNum', 'day', 'chord', 'block_type', 'trial_num_within_chord',
       'num_targets'],
      dtype='str')

In [5]:
subjs = subjs[['subNum', 'BN', 'TN', 'trial_num_within_chord', 'TotalTrialNum', 'targetForce1', 'targetForce2', 'targetForce3', 'targetForce4', 'targetForce5',
            'endForce1', 'endForce2', 'endForce3', 'endForce4', 'endForce5', 'isTargetVisible',
            'endForcePurturbed1', 'endForcePurturbed2', 'endForcePurturbed3', 'endForcePurturbed4', 'endForcePurturbed5',
            'purturbation1', 'purturbation2', 'forceGain','trialCorr', 'trialErrorType', 'num_targets',
            'chord', 'day', 'block_type']]

In [6]:
subjs.head()

,subNum,BN,TN,trial_num_within_chord,TotalTrialNum,targetForce1,targetForce2,targetForce3,targetForce4,targetForce5,...,endForcePurturbed5,purturbation1,purturbation2,forceGain,trialCorr,trialErrorType,num_targets,chord,day,block_type
0,1,1,1,1,1,0.0,-2.0,0.0,-2.0,0.0,...,-6.277439e+66,0.0,0.0,1.0,0,2,2,-v-v-,1,unpurturbed
1,1,1,2,2,2,0.0,-2.0,0.0,-2.0,0.0,...,-1.170000e-01,0.0,0.0,1.0,1,0,2,-v-v-,1,unpurturbed
2,1,1,3,3,3,0.0,-2.0,0.0,-2.0,0.0,...,8.800000e-02,0.0,0.0,1.0,1,0,2,-v-v-,1,unpurturbed
3,1,1,4,4,4,0.0,-2.0,0.0,-2.0,0.0,...,-3.200000e-02,0.0,0.0,1.0,1,0,2,-v-v-,1,unpurturbed
4,1,1,5,5,5,0.0,-2.0,0.0,-2.0,0.0,...,2.000000e-03,0.0,0.0,1.0,1,0,2,-v-v-,1,unpurturbed


In [7]:
subjs[['endForce1', 'endForce2', 'endForce3', 'endForce4', 'endForce5', 
'endForcePurturbed1', 'endForcePurturbed2', 'endForcePurturbed3', 'endForcePurturbed4', 'endForcePurturbed5',
'purturbation1', 'purturbation2', 'num_targets', 'block_type', 'chord', 'day']].head()

,endForce1,endForce2,endForce3,endForce4,endForce5,endForcePurturbed1,endForcePurturbed2,endForcePurturbed3,endForcePurturbed4,endForcePurturbed5,purturbation1,purturbation2,num_targets,block_type,chord,day
0,-6.277439e+66,-6.277439e+66,-6.277439e+66,-6.277439e+66,-6.277439e+66,-6.277439e+66,-6.277439e+66,-6.277439e+66,-6.277439e+66,-6.277439e+66,0.0,0.0,2,unpurturbed,-v-v-,1
1,-4.240000e-01,-6.590000e-01,5.970000e-01,-4.830000e-01,-1.170000e-01,-4.240000e-01,-6.590000e-01,5.970000e-01,-4.830000e-01,-1.170000e-01,0.0,0.0,2,unpurturbed,-v-v-,1
2,-1.570000e-01,-2.482000e+00,7.180000e-01,-1.699000e+00,8.800000e-02,-1.570000e-01,-2.482000e+00,7.180000e-01,-1.699000e+00,8.800000e-02,0.0,0.0,2,unpurturbed,-v-v-,1
3,-2.500000e-02,-1.832000e+00,7.900000e-01,-1.737000e+00,-3.200000e-02,-2.500000e-02,-1.832000e+00,7.900000e-01,-1.737000e+00,-3.200000e-02,0.0,0.0,2,unpurturbed,-v-v-,1
4,6.900000e-02,-1.480000e+00,5.500000e-01,-1.485000e+00,2.000000e-03,6.900000e-02,-1.480000e+00,5.500000e-01,-1.485000e+00,2.000000e-03,0.0,0.0,2,unpurturbed,-v-v-,1


In [8]:
# aligned_cut_force.to_csv(utils.path_misc+'aligned_cut_force.csv', index = False, sep = '\t')

subjs.to_csv(utils.path_misc+'subjs.csv', index = False, sep = '\t')